# 1 - Wczytanie danych

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



# 2 - Podstawowe agregacje

## 2.1 - Liczba transakcji i suma przychodów per sklep

In [6]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



## 2.2 - Statystyki per kategoria

In [7]:
from pyspark.sql.functions import min as _min, max as _max

category_summary = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
        _min("amount").alias("min_transakcja_PLN"),
        _max("amount").alias("max_transakcja_PLN"),
    )
    .orderBy("category")
)
category_summary.show()

+-----------+---------+----------+-----------+------------------+------------------+
|   category|liczba_tx|  suma_PLN|srednia_PLN|min_transakcja_PLN|max_transakcja_PLN|
+-----------+---------+----------+-----------+------------------+------------------+
|elektronika|     2542|1520770.69|     598.26|               9.0|            9999.0|
|    książki|     2574| 851382.08|     330.76|               5.0|           9107.25|
|     odzież|     2453| 849877.55|     346.46|               5.0|           9696.63|
|    żywność|     2431| 789514.43|     324.77|               5.0|           6916.92|
+-----------+---------+----------+-----------+------------------+------------------+



# 3 - Okna tumbling

## 3.1 - Liczba transakcji per godzina (tumbling 1h)

In [8]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [9]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



## 3.2 - Okna 30-minutowe per sklep

In [10]:
half_hour = (
    df.groupBy(window("timestamp", "30 minutes"),"store")   
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("window")
)
half_hour.show(truncate=False)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Wrocław |612      |229985.47|
|2026-

## 3.3 - W której godzinie sklep “Kraków” miał najwyższy przychód?

In [11]:
from pyspark.sql.functions import desc

krakow = (
    df.filter(col("store") == "Kraków")
    .groupBy("store",window("timestamp", "1 hour"))   
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy(desc("suma_PLN"))
)
krakow.show(truncate=False)

+-------------------+-------------------+------+---------+---------+
|od                 |do                 |store |liczba_tx|suma_PLN |
+-------------------+-------------------+------+---------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|Kraków|1169     |483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|Kraków|821      |341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|Kraków|532      |201259.26|
+-------------------+-------------------+------+---------+---------+



# 4 - Okna sliding

## 4.1 - Okno 1h, krok 30 minut

In [12]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



## 4.2 - Tumbling vs sliding

In [13]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: Ponieważ w przypadku okien sliding, okna na siebie nachodzą

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


# 5 - Pytania kontrolne

In [14]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661  

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") prezentuje dane dla każdego sklepu w jednym wierszu, natomiast groupBy(window(...),"store") grupuje dla każdej godziny i każdą godzinę dzieli jeszcze na każdy sklep

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    ODPOWIEDŹ: 2 okja zawierają transakcje z godziny 9:30

# Zadania domowe

In [17]:
from pyspark.sql.functions import asc

gdansk = (
    df.filter(col("store") == "Gdańsk")
    .groupBy("store",window("timestamp", "1 hour"))   
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(avg("amount"), 2).alias("średnia_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "średnia_PLN",
    )
    .orderBy(asc("średnia_PLN"))
    .limit(1)
)
gdansk.show(truncate=False)

+-------------------+-------------------+------+---------+-----------+
|od                 |do                 |store |liczba_tx|średnia_PLN|
+-------------------+-------------------+------+---------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|Gdańsk|766      |395.01     |
+-------------------+-------------------+------+---------+-----------+



In [20]:
from pyspark.sql.functions import hour, minute

wynik_kategorie = (
    df.filter((hour("timestamp") == 9) & (minute("timestamp") < 30))
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx"))
)

wynik_kategorie.show(truncate=False)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|książki    |622      |
|elektronika|611      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+



In [23]:
from pyspark.sql.functions import window, count, desc

max_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx"))
    .limit(1)
)

max_15min.show(truncate=False)

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
+------------------------------------------+---------+

